# Étape 5 — Renouvellements, relocations et immeuble d'Ottawa

Nous mesurons la croissance annualisée du **loyer effectif des mêmes appartements**, en distinguant les renouvellements et les relocations. Les croissances contractuelle et effective sont présentées pour chaque groupe.

- `sRenewal = 1` : renouvellement, le locataire reste.
- `sRenewal = 0` : relocation, un nouveau locataire arrive.

Le statut retenu est celui du **nouveau bail** de chaque paire. Les immeubles québécois et The Met (Ottawa, Ontario) sont analysés séparément. Ce notebook fonctionne indépendamment des autres notebooks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

leases = pd.read_csv("equinoxe_lease_history.csv", dtype={"sPropCode": str, "sUnitCode": str})
leases["sLeaseFrom"] = pd.to_datetime(leases["sLeaseFrom"], errors="coerce")
for col in ["sRent", "sRentEffective", "sRenewal"]:
    leases[col] = pd.to_numeric(leases[col], errors="coerce")
leases = leases.drop_duplicates().copy()
leases["annee"] = leases.sLeaseFrom.dt.year
leases["type_bail"] = leases.sRenewal.map({1: "Renouvellement", 0: "Relocation"})
leases["region_analyse"] = leases.sState.map({"Quebec": "Québec", "Ontario": "Ontario — The Met"})
display(leases[["sBuilding", "sCity", "sState", "region_analyse"]].drop_duplicates())
assert leases.loc[leases.sState.eq("Ontario"), "sBuilding"].eq("The Met").all(), "Autre immeuble ontarien : adapter les groupes."
print("Types de baux ou régions non reconnus :", (leases.type_bail.isna() | leases.region_analyse.isna()).sum())

## 1. Construire les mêmes paires pour les deux loyers

La clé est `sPropCode` + `sUnitCode`, comme demandé dans la consigne. Nous apparions les baux consécutifs avant de filtrer leurs valeurs, afin de ne pas sauter un bail intermédiaire invalide. Nous retenons les durées strictement comprises entre 0,5 et 2,5 ans et les loyers strictement positifs.

Le nombre de paires est le nombre de comparaisons admissibles entre deux baux, pas nécessairement le nombre d'appartements distincts.

In [ ]:
UNIT_KEY = ["sPropCode", "sUnitCode"]
valid_key = leases[UNIT_KEY].notna().all(axis=1)
base = leases.loc[valid_key & leases.sLeaseFrom.notna()].copy()
if base.duplicated(UNIT_KEY + ["sLeaseFrom"]).any():
    raise ValueError("Plusieurs baux pour un appartement à la même date : examiner les doublons.")
base = base.sort_values(UNIT_KEY + ["sLeaseFrom"])
groupes = base.groupby(UNIT_KEY)
for col in ["sRent", "sRentEffective", "sLeaseFrom"]:
    base["precedent_" + col] = groupes[col].shift(1)
base["duree_annees"] = (base.sLeaseFrom - base.precedent_sLeaseFrom).dt.days / 365.25
prix = ["sRent", "sRentEffective", "precedent_sRent", "precedent_sRentEffective"]
admissible = (base[prix].gt(0).all(axis=1)
              & np.isfinite(base[prix]).all(axis=1)
              & base.duree_annees.between(0.5, 2.5, inclusive="neither")
              & base.annee.between(2021, 2025)
              & base.type_bail.notna() & base.region_analyse.notna())
paires = base.loc[admissible].copy()
for prix_col, sortie in [("sRent", "croissance_contractuelle_pct"),
                        ("sRentEffective", "croissance_effective_pct")]:
    paires[sortie] = 100 * ((paires[prix_col] / paires["precedent_" + prix_col]) ** (1 / paires.duree_annees) - 1)
print(f"{len(paires):,} paires admissibles sur {len(leases):,} baux.")
display(paires[UNIT_KEY + ["sBuilding", "annee", "type_bail", "croissance_contractuelle_pct", "croissance_effective_pct"]].head().round(2))

## 2. Comparer les quatre groupes par année

Les hausses sont des **médianes de variations individuelles annualisées**, et non des variations de médianes annuelles.

In [ ]:
def resumer(df, cles):
    resultat = df.groupby(cles).agg(
        nombre_paires=("sRent", "size"),
        croissance_contractuelle_mediane_pct=("croissance_contractuelle_pct", "median"),
        croissance_effective_mediane_pct=("croissance_effective_pct", "median"),
    )
    resultat["ecart_croissance_points"] = (resultat.croissance_contractuelle_mediane_pct
                                           - resultat.croissance_effective_mediane_pct)
    return resultat

resume = resumer(paires, ["region_analyse", "type_bail", "annee"])
display(resume.round(2))
print("Détail par immeuble, pour examiner les différences au sein du Québec :")
par_immeuble = resumer(paires, ["sBuilding", "type_bail", "annee"])
display(par_immeuble.round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
for i, region in enumerate(["Québec", "Ontario — The Met"]):
    for j, type_bail in enumerate(["Renouvellement", "Relocation"]):
        ax = axes[i, j]
        try:
            donnees = resume.xs((region, type_bail), level=("region_analyse", "type_bail"))
        except KeyError:
            ax.text(0.5, 0.5, "Aucune paire admissible", ha="center", transform=ax.transAxes)
        else:
            ax.plot(donnees.index, donnees.croissance_contractuelle_mediane_pct, marker="o", label="Contractuel")
            ax.plot(donnees.index, donnees.croissance_effective_mediane_pct, marker="o", label="Effectif")
            ax.legend()
        ax.axhline(0, color="gray", linewidth=1)
        ax.set(title=f"{region} — {type_bail}", xlabel="Année du nouveau bail", ylabel="Croissance médiane annualisée (%)")
        ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Traiter The Met selon les règles ontariennes

**Ne pas appliquer automatiquement le plafond ontarien à tous les baux.** Le régime dépend notamment de la date de première occupation résidentielle et du changement ou non de locataire.

| Situation | Conséquence pour notre analyse |
|---|---|
| Renouvellement dans un logement assujetti | Examiner la ligne directrice de l'année concernée et les exceptions applicables. |
| Renouvellement dans un immeuble neuf exempté | L'exemption peut retirer le plafond annuel; les obligations de préavis et de fréquence demeurent. |
| Nouveau locataire | Le taux directeur ne limite généralement pas le loyer initial négocié pour la nouvelle location. |

Le gouvernement de l'Ontario indique que les nouveaux immeubles occupés pour la première fois à des fins d'habitation après le **15 novembre 2018** sont exemptés du contrôle des loyers. La Commission précise qu'un préavis de **90 jours** et un intervalle d'au moins **12 mois** restent requis pour les augmentations des locataires en place dans ces logements exemptés.

Sources officielles : [Augmentations des loyers résidentiels — Ontario](https://www.ontario.ca/fr/page/augmentations-des-loyers-residentiels) et [Exemption et obligations — Commission de la location immobilière](https://tribunalsontario.ca/2019/01/24/ltb-news-release-rent-control-changes-to-the-residential-tenancies-act/). Sources consultées le 3 octobre 2026.

**Application à The Met :** le notebook de départ décrit une mise en service récente, en 2023–2024, ce qui suggère une exemption si cette information correspond bien à la première occupation résidentielle de l'immeuble. Le premier bail observé dans cet extrait ne suffit pas à prouver cette date. Le statut reste à confirmer à partir d'une source ou d'un document fiable.

Ces règles concernent le loyer légal; `sRentEffective` ne constitue pas à lui seul une mesure du loyer légal. Les croissances annualisées et médianes présentées ne permettent pas non plus de conclure à la conformité d'un bail. Nous conservons les observations et n'écrêtons pas automatiquement les hausses à un plafond.

Pour éviter d'utiliser des informations futures dans le backtest, retenir uniquement les règles et annonces connues à la date de chaque prévision.

In [ ]:
the_met = leases.loc[leases.sBuilding.eq("The Met")]
display(pd.DataFrame({
    "Information": ["Premier début de bail observé dans l'extrait", "Statut au regard du plafond"],
    "Résultat": [str(the_met.sLeaseFrom.min().date()), "À confirmer : première occupation résidentielle à documenter"],
}))

## 4. Préparer l'explication et la prévision

À partir des tableaux et graphiques :

1. Comparer les hausses effectives des renouvellements et des relocations, en citant les années et les effectifs.
2. Examiner si The Met suit une tendance différente des immeubles québécois; ne pas attribuer cette différence uniquement à la réglementation.
3. Pour 2026, prévoir séparément les groupes avant toute combinaison. Justifier les poids choisis; une moyenne pondérée de hausses moyennes ne doit pas être présentée comme une médiane combinée.

Texte de méthode utilisable dans le rapport :

> Nous apparions les baux consécutifs de chaque appartement à l'aide de `sPropCode` et `sUnitCode`. Nous classons chaque comparaison selon le statut du nouveau bail et distinguons les renouvellements des relocations. Nous isolons The Met, situé en Ontario, afin de tenir compte du régime provincial applicable. L'éventuelle exemption au plafond doit être confirmée; elle ne peut pas être déduite du seul historique des baux de l'extrait.

Les règles du Québec seront documentées avec les sources du TAL à l'étape 6. Cet extrait ne permet pas d'estimer l'occupation ou le revenu total du portefeuille.